# Bias and Variance Tradeoff - Overfitting

2025-06-16 ug V1.6

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Erzeugung von Daten (X,y) 
Die hier erzeugten Daten (X, y) weisen einen nichtlinearen Zusammenhang auf.

Wir wählen einen störungsfreien, deterministischen und nichtlinearen Zusammenhang zwischen dem Eingang $X$ und Ausgang $y_{org}$ :

$
y_{org}(X) = 6 \cdot \sin{\left(\frac{\pi \cdot X}{2*8}\right)}
$

Am Ausgang $y$ überlagern wir eine normalverteilte und mit dem Ausgang $y_{org}$ unkorrelierte Störung $\epsilon_{org}$.

$
y(X) = y_{org}(X) + \epsilon 
$

mit  $\epsilon_{org} \sim \mathcal{N}(\mu,\,\sigma^{2})$, Erwartungswert $\mu=0$ (mittelwertfrei) und konstanter Varianz $\sigma^2=1$.


In [ ]:
N=17                                              # Anzahl der Datenpunkte
x_max = 8                                         # maximaler Wert für den Eingang
np.random.seed(42)                                # Initialisieren des Zufallgenerators
x_org = np.linspace(0,x_max,N)                    # Eingang mit gleiche Abständnen im Bereich 0 und x_max
y_org_nominal = 6.0 * np.sin(x_org/x_max*np.pi/2) # ungestörter Ausgang
epsilon_org = 0.5*np.random.randn(N)              # Störung
y_org = y_org_nominal + epsilon_org               # Störung dem Ausgang überlagern
X = x_org.reshape((-1,1))                         # X als Matrix anlegen
y = y_org                                         # Ausgang

In [ ]:
X,y

**Anmerkung:** In der Praxis gibt es meist mehrere Eingänge und nicht wie hier nur einen Eingang. Deshalb ist die X als 2-dimensionale Array (Matrix) angelegt. 

----
### Plotten der Daten

In [ ]:
plt.plot(X, y,marker = 'o',ls='None',label='Datenpunkte',color='black')
plt.plot(x_org, y_org_nominal,label='true behavior',color='orange')
plt.xlabel('unabhängige Variable')
plt.ylabel('anhängige Variable')
plt.grid(True)
plt.legend();

---
### Aufteilen der Daten in einen Trainings- und einen Testdatensatz

Dazu stellt scikit-learn die Funktion 
[`sklearn.model_selection.train_test_split()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
bereit. 


In [ ]:
#from sklearn.model_selection import train_test_split
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

Zur besseren grafischen Visualisierung verwenden wir hier aber einen einfacheren Ansatz:
Trainingsdaten wechseln sich mit Testdaten ab.

In [ ]:
X_train, X_test, y_train, y_test = X[::2],X[1::2], y[::2],y[1::2]

In [ ]:
X_train.shape, X_test.shape

Plotten des Trainings und Testdatensatzes:

In [ ]:
plt.plot(X_train, y_train,marker = 'D',ls='None', label='train', color='green')
plt.plot(X_test, y_test,marker = 's',ls='None',label='test',color='magenta')
plt.plot(x_org, y_org_nominal,label='true behavior',color='orange')
plt.legend()
plt.grid(True)

---
## Lineare Regression

Modellannahme bei der Linearen Regression:
$
y_i(x_i) = \beta_0 + \beta_1 x_i + \epsilon_i
$

mit $i=0 .. N-1$
- $\beta_0$ - wahrer Achsenabschnitt (true intercept)
- $\beta_1$ - wahre Steigung (true slope)
- $\epsilon_i$ - Störung unkorreliert und normalverteilt $\sim \mathcal{N}(\mu,\,\sigma^{2})$ mit Erwartungswert $\mu=0$ (mittelwertfrei) und konstanter Varianz $\sigma^2$

Prädiktions-Modell bzw. Schätzer:
$
\hat{y}_i(x_i) = \hat{\beta}_0 + \hat{\beta}_1 x_i
$
mit den Modellparametern:
- $\hat{\beta}_0$ - geschätzter Achsenabschnitt (estimated intercept)
- $\hat{\beta}_1$ - geschätzte Steigung (estimated slope)

Mit Hilfe der Klasse 
[`sklearn.linear_model.LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- bestimmen wir die Modellparameter $\hat{\beta}_0$ und $\hat{\beta}_1$ und 
- bestimmen wir die Leistungsfähigkeit des Prädiktions-Modells mittels des Bestimmungsmass (coefficient of determination) **$R^2$**:

Das Bestimmungsmass (coefficient of determination) **$R^2$** ermitteln wir mittels folgender Methoden/Funktionen:
   - [`score(X, y)`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression.score)
   - [`metrics.r2_score(y,y_hat)`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)



In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn import metrics

Anpassen der Modellparameter der Linearen Regression $\hat{\beta}_0$ und $\hat{\beta}_1$ durch den Trainingsdatensatz:

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

Ausgabe der geschätzen Modellparameter:

In [ ]:
print(f"geschätzte Steigung (estimated slope)            : {lr_model.coef_[0]:.3f}")
print(f"geschätzter Achsenabschnitt (estimated intercept): {lr_model.intercept_:.3f}")

Bestimmung der Leistungsfähigkeit der Methode [`score(X, y)`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression.score)

In [ ]:
print(f"R_squared: {lr_model.score(X_train, y_train):.3f}")

#### Bestimmungsmass/Determinationskoeffizient $R^2$

Kennzahl zur Beurteilung der Anpassungsgüte einer Regression 
[Bestimmungsmass-Wikipedia](https://de.wikipedia.org/wiki/Bestimmtheitsma%C3%9F)

Das Bestimmungsmass $R^2$ gibt an, wie viel Streuung in den Daten durch ein vorliegendes lineares Regressionsmodell „erklärt“ werden kann.

Interpretation:

Das Bestimmtheitsmaß lässt sich mit 100 % multiplizieren, um es in Prozent anzugeben: 100 % $R^2$ ist dann der prozentuale Anteil der Streuung in $y$ der durch das lineare Modell „erklärt“ wird, und liegt daher zwischen:
-   0 % (oder 0): schlechtes Modell und
- 100 % (oder 1): perfektes Modell.

[`sklearn.metrics.r2_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)

Prädiktion des Traingsdatensatzes


In [ ]:
y_pred_LinearRegression_train = lr_model.predict(X_train)
y_pred_LinearRegression_train

In [ ]:
R_squared = metrics.r2_score(y_train,y_pred_LinearRegression_train)
R_squared

Mean Squared Error
$
\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
$

[`sklearn.metrics.mean_squared_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

In [ ]:
MSE = metrics.mean_squared_error(y_train, y_pred_LinearRegression_train)
MSE

#### Plot der Prädiktion

In [ ]:
# Linie des Prädiktionsmodells erzeugen
X_plot = np.linspace(0,x_max,200).reshape(-1,1)
y_plot_pred_LinearRegression_train = lr_model.predict(X_plot)

für die Trainingsdaten

In [ ]:
plt.plot(x_org, y_org_nominal,label='true behavior',color='orange')
plt.plot(X_train, y_train, marker = 'D',ls='None', label='train data',color='green')
plt.plot(X_train, y_pred_LinearRegression_train,marker = 'o',ls='None',label='pred data',color='red')
plt.plot(X_plot, y_plot_pred_LinearRegression_train,label='lineare Regression',linestyle=':',color='red')

plt.legend()
plt.grid(True)

für den Testdatensatz

In [ ]:
print(f"R_squared: {lr_model.score(X_test, y_test):.3f}")

In [ ]:
y_pred_LinearRegression_test = lr_model.predict(X_test)
y_pred_LinearRegression_test

In [ ]:
R_squared = metrics.r2_score(y_test,y_pred_LinearRegression_test)
R_squared

In [ ]:
MSE = metrics.mean_squared_error(y_test, y_pred_LinearRegression_test)
MSE

In [ ]:
plt.plot(x_org, y_org_nominal,label='true behavior',color='orange')
#plt.plot(X_train, y_train, marker = 'D',ls='None', label='train data',color='green')
plt.plot(X_test, y_test, marker = 's',ls='None', label='train data',color='magenta')
plt.plot(X_test, y_pred_LinearRegression_test,marker = 'o',ls='None',label='pred data',color='red')
plt.plot(X_plot, y_plot_pred_LinearRegression_train,label='lineare Regression',linestyle=':',color='red')
plt.legend()
plt.grid(True)

---
## Polynomial Regression

$
y_i(x_i) = \beta_0 + \beta_1 x_i + \beta_2 x_i^2 + \beta_3 x_i^3 + ...  + \beta_p x_i^p + \epsilon_i
$

mit $i=0 .. N-1$ und $p$ Polynomgrad
- Parametern $\beta_0$, $\beta_1$, $\beta_2$, $\beta_3$ .. , $\beta_p$ 
- $\epsilon_i$ - unkorreliert und normalverteilt $\sim \mathcal{N}(\mu,\,\sigma^{2})$ mit Erwartungswert $\mu=0$ (mittelwertfrei) und konstanter Varianz $\sigma^2$

Präditionsmodell bzw. Schätzer:

$
\hat{y}_i(x_i) = \hat{\beta}_0 + \hat{\beta}_1 x_i + \hat{\beta}_2 x_i^2 + \hat{\beta}_3 x_i^3 + ...  + \hat{\beta}_p x_i^p 
$
und den Modellparametern:
- $\hat{\beta}_0$, $\hat{\beta}_1$, $\hat{\beta}_2$, $\hat{\beta}_3$ und $\hat{\beta}_p$ 



Ist immer noch eine "lineare Regression", weil die zu schätzenden Parameter $\beta_0$, $\beta_1$, $\beta_2$, $\beta_3$ .. , $\beta_p$ linear in das Modell eingehen.

Aufbau der Features mit der Klasse [`sklearn.preprocessing.PolynomialFeatures()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

Datensatz für lineare Regression der Polynomkoeffizienten aufbauen

In [ ]:
X_train

In [ ]:
X_train.shape

 ### Polynom 8ten Grades
 Wir wählen ein Polynom 8ten Grades:

In [ ]:
pre_process_poly8 = PolynomialFeatures(degree=8,include_bias=False)

X_train_poly8 = pre_process_poly8.fit_transform(X_train)
X_train_poly8

In [ ]:
X_train_poly8.shape

Objekt für Berechnung der Lineare Regression erstellen

In [ ]:
pr_model_poly8 = LinearRegression()

Modell an die Daten anpassen

In [ ]:
pr_model_poly8.fit(X_train_poly8, y_train)

In [ ]:
pr_model_poly8.intercept_, pr_model_poly8.coef_ 

Prädiktion

In [ ]:
y_poly8_pred_train = pr_model_poly8.predict(X_train_poly8)

Bestimmtheitsmass $R^2$

Modell passt perfekt zu den angefitteten Daten

In [ ]:
R_squared_poly8 = metrics.r2_score(y_train,y_poly8_pred_train)
R_squared_poly8

-> **Perfekt!**

Mean Squared Error 

ist auch extrem klein

In [ ]:
MSE = metrics.mean_squared_error(y_train, y_poly8_pred_train)
MSE

#### Plot der Prädiktion

für den Trainingsdatensatz

In [ ]:
X_plot_train_poly = pre_process_poly8.fit_transform(X_plot)
y_plot_pred_PolyRegression_train = pr_model_poly8.predict(X_plot_train_poly)

In [ ]:
plt.plot(x_org, y_org_nominal,label='true',color='orange')
plt.plot(X_train, y_train, marker = 'D',ls='None', label='train',color='green')
plt.plot(X_train, y_poly8_pred_train,marker = 'o',ls='None',label='pred',color='red')
plt.plot(X_plot, y_plot_pred_PolyRegression_train,label='pred',color='red',linestyle=':')

plt.legend()
plt.ylim(0,7)
plt.xlim(-1,10)
plt.grid(True)




für den Testdatensatz

In [ ]:
X_test_poly8 = pre_process_poly8.fit_transform(X_test)
X_test_poly8

In [ ]:
print(f"R_squared: {pr_model_poly8.score(X_test_poly8, y_test):.3f}")

In [ ]:
y_pred_poly8_test = pr_model_poly8.predict(X_test_poly8)
y_pred_poly8_test

In [ ]:
R_squared = metrics.r2_score(y_test,y_pred_poly8_test)
R_squared

In [ ]:
MSE = metrics.mean_squared_error(y_test, y_pred_poly8_test)
MSE

In [ ]:
plt.plot(x_org, y_org_nominal,label='true behavior',color='orange')
#plt.plot(X_train, y_train, marker = 'D',ls='None', label='train data',color='green')
plt.plot(X_test, y_test, marker = 's',ls='None', label='train data',color='magenta')
plt.plot(X_test, y_pred_poly8_test,marker = 'o',ls='None',label='pred data',color='red')
plt.plot(X_plot, y_plot_pred_PolyRegression_train,label='pred',color='red',linestyle=':')
plt.legend()
plt.grid(True)

## Schleife über Polynome 1 bis 8 Ordnung

In [ ]:
Liste_Ordnung   = []
Liste_MSE_train = []
Liste_MSE_test  = []

for Ordnung in range(1,9):
    print(f"Polynom {Ordnung}-ter Ordnung")
    
    # Feature erzeugen 
    pre_process_poly = PolynomialFeatures(degree=Ordnung,include_bias=False)
    X_train_poly = pre_process_poly.fit_transform(X_train)
    X_test_poly  = pre_process_poly.fit_transform(X_test)
    
    # Polynomial Regression
    pr_model = LinearRegression()
    pr_model.fit(X_train_poly, y_train)
    
    # Prädiktion mit Trainingsdaten
    y_poly_pred_train = pr_model.predict(X_train_poly)
    
    # Prädiktion mit Testdaten
    y_poly_pred_test = pr_model.predict(X_test_poly)
    
    # Mean Square Error für Trainings und Testdaten
    MSE_train = metrics.mean_squared_error(y_train, y_poly_pred_train)
    MSE_test  = metrics.mean_squared_error(y_test, y_poly_pred_test)

    Liste_Ordnung.append(Ordnung)
    Liste_MSE_train.append(MSE_train)
    Liste_MSE_test.append(MSE_test)


In [ ]:
data = {'MSE_train': Liste_MSE_train, 'MSE_test': Liste_MSE_test}
df_MSE = pd.DataFrame(data=data,index=Liste_Ordnung)
df_MSE

In [ ]:
fig, ax = plt.subplots()
df_MSE.plot(ax=ax,marker='x');
ax.grid(True)
ax.set_xlabel('Modell-Komplexität - Ordnung des Polynom')
ax.set_ylabel('Fehler - MSE')
ax.set_title('Fehler versus Modell Komplexität');

Interpretation des Diagramms:
<br>
- Vom linearem Modell erster Ordnung zum Polynom zweiter Ordnung gibt es eine deutliche Verbesserung von ca. 0.4 bis 0.12.
- Dann gibt es vom Polynom 2ter Ordnung bis zum Polynom 5ter einen sehr flachen Verlauf des Fehlers (Plateau).
- Ab dem Polynom 6ter Ordnung steigt der Fehler mit dem Test-Datensatz deulich an. Hier beginnt das _**Overfitting**_.
Der Fehler mit dem Test-Datensatz wird immer kleiner und der Fehler mit dem Test-Datensatz steigt stark an.

In der Praxis wird ein _**Modell mit Polynom 2ter Ordnung**_ verwendet werden, weil es am Anfang des Plateau liegt.
<br>
Im folgenden werden wir uns das Modell mit Polynom 2ter Ordnung näher ansehen:

 ### Polynom 2ten Grades
 Wir wählen ein Polynom 2ten Grades:

In [ ]:
pre_process_poly2 = PolynomialFeatures(degree=2,include_bias=False)

X_train_poly2 = pre_process_poly2.fit_transform(X_train)
X_train_poly2

In [ ]:
X_train_poly2.shape

Objekt für Berechnung der Lineare Regression erstellen

In [ ]:
pr_model_poly2 = LinearRegression()

Modell an die Daten anpassen

In [ ]:
pr_model_poly2.fit(X_train_poly2, y_train)

In [ ]:
pr_model_poly2.intercept_, pr_model_poly2.coef_ 

Prädiktion

In [ ]:
y_poly2_pred_train = pr_model_poly2.predict(X_train_poly2)

Bestimmtheitsmass $R^2$

Modell passt perfekt zu den angefitteten Daten

In [ ]:
R_squared_poly2 = metrics.r2_score(y_train,y_poly2_pred_train)
R_squared_poly2

-> **sehr gut!**

Mean Squared Error 

ist auch extrem klein

In [ ]:
MSE = metrics.mean_squared_error(y_train, y_poly2_pred_train)
MSE

#### Plot der Prädiktion

für den Traingsdatensatz

In [ ]:
X_plot_train_poly = pre_process_poly2.fit_transform(X_plot)
y_plot_pred_PolyRegression_train = pr_model_poly2.predict(X_plot_train_poly)

In [ ]:
plt.plot(x_org, y_org_nominal,label='true',color='orange')
plt.plot(X_train, y_train, marker = 'D',ls='None', label='train',color='green')
plt.plot(X_train, y_poly2_pred_train,marker = 'o',ls='None',label='pred',color='red')
plt.plot(X_plot, y_plot_pred_PolyRegression_train,label='pred',color='red',linestyle=':')

plt.legend()
plt.ylim(0,7)
plt.xlim(-1,10)
plt.grid(True)

für den Testdatensatz

In [ ]:
X_test_poly2 = pre_process_poly2.fit_transform(X_test)
X_test_poly2

In [ ]:
print(f"R_squared: {pr_model_poly2.score(X_test_poly2, y_test):.3f}")

In [ ]:
y_pred_poly2_test = pr_model_poly2.predict(X_test_poly2)
y_pred_poly2_test

In [ ]:
R_squared = metrics.r2_score(y_test,y_pred_poly2_test)
R_squared

In [ ]:
MSE = metrics.mean_squared_error(y_test, y_pred_poly2_test)
MSE

In [ ]:
plt.plot(x_org, y_org_nominal,label='true behavior',color='orange')
#plt.plot(X_train, y_train, marker = 'D',ls='None', label='train data',color='green')
plt.plot(X_test, y_test, marker = 's',ls='None', label='train data',color='magenta')
plt.plot(X_test, y_pred_poly2_test,marker = 'o',ls='None',label='pred data',color='red')
plt.plot(X_plot, y_plot_pred_PolyRegression_train,label='pred',color='red',linestyle=':')
plt.legend()
plt.grid(True)

Anmerkung: Das Modell mit dem Polynom 2ter Ordnung gibt schon sehr gut das tatsächliche Verhaltn wieder.